In [1]:
import gradio as gr
import requests

# Catalog maps the display name to the ID used by the backend
CATALOG = {
    "Airpods": "p1", 
    "Charging Data Cable": "p2", 
    "Mobile Phone": "p3",
    "Micro SD Memory Card": "p4", 
    "FOAM Cleaning Agent": "p5", 
    "WIFI Adapter": "p6",
    "HDMI Cable": "p7", 
    "Car Phone Holder": "p8", 
    "Car Charger": "p9",
    "Microphone": "p10", 
    "USB Flash Drive": "p11", 
    "Handsfree Earphones": "p12",
    "LED Ring Light": "p13", 
    "Phone Cover": "p14", 
    "Selfie Stick": "p15",
    "Wireless Keyboard": "p16", 
    "Mini Speaker": "p17", 
    "Bike Phone Holder": "p18",
    "Laptop Battery": "p19"
}

# --- FUNCTIONS ---

def add_to_cart(cart, product_name, quantity):
    """Adds selected items to the cart state"""
    if not product_name or quantity <= 0:
        return cart, "⚠️ Please select a product and quantity."
    
    item_id = CATALOG.get(product_name)
    
    # Add to cart list (we store the ID multiple times for simple processing)
    # e.g., 2 Airpods -> ["p1", "p1"]
    new_items = [item_id] * int(quantity)
    cart.extend(new_items)
    
    # Generate display text
    display_text = f"🛒 Current Cart ({len(cart)} items):\n"
    # Helper to show what's in cart nicely
    from collections import Counter
    readable_cart = [key for key, val in CATALOG.items() if val in cart]
   
    # This part is tricky because 'cart' is just IDs. 
    # Let's count IDs first
    id_counts = Counter(cart)
    # Map back to names
    reverse_catalog = {v: k for k, v in CATALOG.items()}
    
    for pid, count in id_counts.items():
        p_name = reverse_catalog.get(pid, "Unknown")
        display_text += f"- {p_name} (x{count})\n"
        
    return cart, display_text

def clear_cart():
    return [], "🗑️ Cart Cleared."

def place_order(name, email, cart):
    """Submits the cart to the backend"""
    if not cart:
        return "⚠️ Cart is empty! Add items first."
    if not name or not email:
        return "⚠️ Please enter Name and Email."
        
    payload = {
        "customer": name,
        "email": email,
        "items": cart # Sends list of IDs: ["p1", "p1", "p3"]
    }
    
    try:
        # Call Order Service
        response = requests.post("http://localhost:5000/place_order", json=payload)
        
        if response.status_code == 200:
            res_data = response.json()
            
            # Use the pre-formatted details from backend
            details_text = "\n".join(res_data['details'])
            
            return (f"✅ ORDER SUCCESSFUL!\n"
                    f"--------------------------------\n"
                    f"Order ID: {res_data['order_id']}\n"
                    f"Customer: {name}\n"
                    f"Total Cost: Rs. {res_data['total']}\n"
                    f"--------------------------------\n"
                    f"Sourcing Details:\n{details_text}\n"
                    f"--------------------------------\n"
                    f"A receipt has been sent to {email}")
        else:
            return f"❌ Order Failed: {response.text}"
            
    except requests.exceptions.ConnectionError:
        return "❌ CONNECTION ERROR: Could not reach Order Service. Is '3_Order.ipynb' running?"

# --- UI BUILDER ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    # State variable to hold the cart (invisible to user, stores data)
    cart_state = gr.State([])

    gr.Markdown("# 🛒 The Gadget Hub Store")
    gr.Markdown("Add multiple items to your cart and place a single order.")
    
    with gr.Row():
        name_input = gr.Textbox(label="Customer Name", placeholder="John Doe")
        email_input = gr.Textbox(label="Email Address", placeholder="john@example.com")
    
    with gr.Row():
        product_dropdown = gr.Dropdown(choices=list(CATALOG.keys()), label="Select Product", value="Airpods")
        qty_input = gr.Number(value=1, label="Quantity", minimum=1, precision=0)
        add_btn = gr.Button("➕ Add to Cart", variant="secondary")
    
    # Cart Display Area
    cart_display = gr.Textbox(label="Your Cart", value="Cart is empty", lines=5, interactive=False)
    clear_btn = gr.Button("🗑️ Clear Cart")
    
    # Submit Area
    place_order_btn = gr.Button("✅ Place Order", variant="primary")
    output_text = gr.Textbox(label="Order Receipt", lines=10)
    
    # Event Handlers
    add_btn.click(
        add_to_cart, 
        inputs=[cart_state, product_dropdown, qty_input], 
        outputs=[cart_state, cart_display]
    )
    
    clear_btn.click(
        clear_cart,
        inputs=None,
        outputs=[cart_state, cart_display]
    )
    
    place_order_btn.click(
        place_order, 
        inputs=[name_input, email_input, cart_state], 
        outputs=output_text
    )

print("Client App Running...")
demo.launch(share=False)

C:\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\User\AppData\Local\Temp\ipykernel_19484\1530452565.py:101: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Client App Running...
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
